In [1]:
%pip install transformers torch pandas scikit-learn joblib matplotlib

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.



   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.1 MB ? eta -:--:--
   --- ------------------------------------ 0.8/10.1 MB 1.3 MB/s eta 0:00:07
   ---- ----------------------------------- 1.0/10.1 MB 1.4 MB/s eta 0:00:07
   ------ --------------------------------- 1.6/10.1 MB 1.6 MB/s eta 0:00:06
   ------- -------------------------------- 1.8/10.1 MB 1.6 MB/s eta 0:00:06
   --------- ------------------------------ 2.4/10.1 MB 1.7 MB/s eta 0:00:05
   ----------- ---------------------------- 2.9/10.1 MB 1.8 MB/s eta 0:00:04
   ------------- -------------------------- 3.4/10.1 MB 1.9 MB/s eta 0:00:04
   --------------- ------------------------ 3.9/10.1 MB 2.0 MB/s eta 0:00:04
   ---------------- ----------------------- 4.2/10.1 MB 2.0 MB/s eta 0:00:04
   ----------------- ---------------------- 4.5/10.1 MB 1.8 MB/s eta 0:00:04
   ----------------

In [4]:
# ==============================
# Cell 1: Imports and Setup
# ==============================

import os  # For handling file paths and directories
import random  # For setting random seeds
import numpy as np  # For numerical operations
import pandas as pd  # For working with tabular data

import torch  # Main PyTorch library
from torch.utils.data import Dataset, DataLoader  # For creating datasets and data loaders

from sklearn.model_selection import train_test_split  # For splitting dataset into train/val/test
from sklearn.preprocessing import LabelEncoder  # For converting labels (strings) to integers
from sklearn.metrics import accuracy_score, classification_report  # For evaluating model performance

from transformers import AutoTokenizer, AutoModelForSequenceClassification  # For loading pre-trained Transformer models
from torch.optim import AdamW # Optimizer
from transformers.optimization import get_linear_schedule_with_warmup  # Learning rate scheduler

from joblib import dump  # For saving the label encoder to disk (serialization)

# Check if a GPU is available; if yes, use it; otherwise, use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Choose computing device

print(f"Using device: {device}")  # Print which device is being used


# ==============================
# Cell 2: Configuration & Paths
# ==============================

# Name of the pre-trained Transformer model to use
MODEL_NAME = "distilbert-base-uncased"  # Lightweight version of BERT

# Maximum number of tokens per input sequence (truncate or pad to this length)
MAX_LEN = 64  # Suitable for short questions

# Training hyperparameters
BATCH_SIZE = 16  # Number of samples per training batch
EPOCHS = 3  # Number of passes over the entire training dataset
LEARNING_RATE = 2e-5  # Initial learning rate for the optimizer

# Random seed for reproducibility
RANDOM_SEED = 42  # Fix seed to make results more repeatable

# Set random seeds for Python, NumPy, and PyTorch
random.seed(RANDOM_SEED)  # Seed for Python's random module
np.random.seed(RANDOM_SEED)  # Seed for NumPy random
torch.manual_seed(RANDOM_SEED)  # Seed for PyTorch (CPU)
if torch.cuda.is_available():  # If GPU is available
    torch.cuda.manual_seed_all(RANDOM_SEED)  # Set seed for all GPU devices


# Create necessary directories if they do not exist
os.makedirs("data", exist_ok=True)  # Folder to store dataset files
os.makedirs("models", exist_ok=True)  # Folder to store trained models
os.makedirs("results", exist_ok=True)  # Folder to store evaluation results and plots


# ==============================
# Cell 3: Loading or Creating Dataset
# ==============================

# Path to the main dataset CSV file (you can replace this with your own dataset)
DATA_PATH = "data/intent_questions.csv"  # CSV file containing questions and labels

# Define the example data with sufficient samples for stratification
example_data = {  # Define a dictionary with sample questions and labels
    "text": [
        "How much urea should I use for wheat per acre?",  # fertilizer_wheat
        "What is the best way to control pests in rice?",  # pest_control_rice
        "How often should I irrigate my maize crop?",  # irrigation_maize
        "What fertilizer is recommended for paddy?",  # fertilizer_rice
        "How can I prevent fungal disease in wheat?",  # disease_control_wheat
        "When should I water tomatoes in summer?",  # irrigation_vegetables
        "How to control insects in cotton crop?",  # pest_control_cotton
        "Recommended fertilizer dose for rice?",  # fertilizer_rice
        "How much water does wheat need in winter?",  # irrigation_wheat
        "What is the best pesticide for maize worms?", # pest_control_maize
        # Added samples to ensure at least 2 per class for stratification
        "What is the recommended fertilizer for wheat crops?", # fertilizer_wheat
        "How do I control pests in my rice field?", # pest_control_rice
        "When should maize be irrigated?", # irrigation_maize
        "What fertilizer should I use for rice?", # fertilizer_rice
        "Fungicide for wheat disease prevention?", # disease_control_wheat
        "Irrigation frequency for tomatoes in hot weather?", # irrigation_vegetables
        "Best insecticides for cotton?", # pest_control_cotton
        "How much water is needed for winter wheat?", # irrigation_wheat
        "Controlling worms in maize crops?" # pest_control_maize
    ],
    "label": [
        "fertilizer_wheat",
        "pest_control_rice",
        "irrigation_maize",
        "fertilizer_rice",
        "disease_control_wheat",
        "irrigation_vegetables",
        "pest_control_cotton",
        "fertilizer_rice",
        "irrigation_wheat",
        "pest_control_maize",
        # Added labels
        "fertilizer_wheat",
        "pest_control_rice",
        "irrigation_maize",
        "fertilizer_rice",
        "disease_control_wheat",
        "irrigation_vegetables",
        "pest_control_cotton",
        "irrigation_wheat",
        "pest_control_maize"
    ]
}
df_example = pd.DataFrame(example_data)  # Convert dictionary to a pandas DataFrame
df_example.to_csv(DATA_PATH, index=False)  # Save the example dataset to CSV (overwrite if exists)
print(f"Created/Updated example dataset at {DATA_PATH}")  # Inform the user that a sample dataset was created/updated

# Now load the dataset from the CSV file
df = pd.read_csv(DATA_PATH)  # Read the dataset into a DataFrame
print(df.head())  # Show the first few rows to verify the data


# ==============================
# Cell 4: Encode Labels and Split Data
# ==============================

# Initialize a LabelEncoder to transform string labels into integer indices
label_encoder = LabelEncoder()  # Create a LabelEncoder instance

# Fit the encoder on the 'label' column and transform it to integer labels
df["label_id"] = label_encoder.fit_transform(df["label"])  # Add a new column with encoded labels

# Show mapping from label string to integer ID
label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))  # Create mapping dict
print("Label mapping:", label_mapping)  # Print label mapping for reference

# Extract features (questions) and labels (encoded IDs)
texts = df["text"].values  # Numpy array of question texts
labels = df["label_id"].values  # Numpy array of integer labels

# First split: train + temp (where temp will later be split into val and test)
X_train, X_temp, y_train, y_temp = train_test_split(
    texts,  # Full set of texts
    labels,  # Full set of labels
    test_size=0.5,  # 50% of data goes to temp (val + test)
    random_state=RANDOM_SEED,  # Use fixed random seed
    stratify=labels  # Keep label distribution similar in splits (if possible)
)

# Second split: split temp into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,  # Temporary feature set
    y_temp,  # Temporary label set
    test_size=0.5,  # Split temp into 50% val and 50% test (so each is 15% of total)
    random_state=RANDOM_SEED  # Fixed random seed
    # stratify=y_temp  # Preserve label distribution -- Removed to avoid error with small classes
)

# Save the splits to separate CSV files for reuse in validation and test notebooks
train_df = pd.DataFrame({"text": X_train, "label_id": y_train})  # Training DataFrame
val_df = pd.DataFrame({"text": X_val, "label_id": y_val})  # Validation DataFrame
test_df = pd.DataFrame({"text": X_test, "label_id": y_test})  # Test DataFrame

# Paths for the split datasets
TRAIN_PATH = "data/train.csv"  # Path to training data CSV
VAL_PATH = "data/val.csv"  # Path to validation data CSV
TEST_PATH = "data/test.csv"  # Path to test data CSV

# Save DataFrames to CSV
train_df.to_csv(TRAIN_PATH, index=False)  # Save train set
val_df.to_csv(VAL_PATH, index=False)  # Save validation set
test_df.to_csv(TEST_PATH, index=False)  # Save test set

print(f"Train samples: {len(train_df)}, Val samples: {len(val_df)}, Test samples: {len(test_df)}")  # Print counts

# Save the label encoder so we can use it later in other notebooks and in chatbot.py
LABEL_ENCODER_PATH = "models/label_encoder.joblib"  # Path to save label encoder
dump(label_encoder, LABEL_ENCODER_PATH)  # Save the label encoder to disk
print(f"Saved label encoder to {LABEL_ENCODER_PATH}")  # Confirm save


# ==============================
# Cell 5: Dataset Class and Tokenizer
# ==============================

# Load the tokenizer corresponding to the chosen Transformer model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  # Load tokenizer for DistilBERT

class IntentDataset(Dataset):  # Define a custom dataset class for intent classification
    def __init__(self, texts, labels, tokenizer, max_len):  # Constructor with texts, labels, tokenizer, and max length
        self.texts = texts  # Store the input texts (questions)
        self.labels = labels  # Store the integer labels
        self.tokenizer = tokenizer  # Store the tokenizer for encoding text
        self.max_len = max_len  # Store maximum sequence length

    def __len__(self):
        return len(self.texts)  # Number of text samples

    def __getitem__(self, idx):  # Get a single item (sample) by index
        text = str(self.texts[idx])  # Get the question text at the given index and ensure it's a string
        label = int(self.labels[idx])  # Get the corresponding label and ensure it's an integer

        # Use the tokenizer to convert text into token IDs and attention mask
        # encode_plus was removed in newer transformers versions; use the tokenizer callable instead
        encoding = self.tokenizer(
            text,  # Text to encode
            add_special_tokens=True,  # Add special tokens like [CLS] and [SEP]
            max_length=self.max_len,  # Truncate/pad to this maximum length
            padding="max_length",  # Pad to max_length
            truncation=True,  # Truncate texts longer than max_len
            return_attention_mask=True,  # Return attention mask (1 for tokens, 0 for padding)
            return_tensors="pt"  # Return PyTorch tensors
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),  # Flatten token IDs tensor to 1D
            "attention_mask": encoding["attention_mask"].flatten(),  # Flatten attention mask tensor
            "labels": torch.tensor(label, dtype=torch.long)  # Create a tensor for the label
        }


# ==============================
# Cell 6: Create DataLoaders
# ==============================

# Create dataset objects for train and validation sets
train_dataset = IntentDataset(
    texts=X_train,  # Training texts
    labels=y_train,  # Training labels
    tokenizer=tokenizer,  # Tokenizer
    max_len=MAX_LEN  # Maximum sequence length
)

val_dataset = IntentDataset(
    texts=X_val,  # Validation texts
    labels=y_val,  # Validation labels
    tokenizer=tokenizer,  # Tokenizer
    max_len=MAX_LEN  # Maximum sequence length
)

# Create DataLoader objects to efficiently batch and shuffle data
train_loader = DataLoader(
    train_dataset,  # Dataset to load from
    batch_size=BATCH_SIZE,  # Batch size for training
    shuffle=True  # Shuffle training data each epoch
)

val_loader = DataLoader(
    val_dataset,  # Dataset to load from
    batch_size=BATCH_SIZE,  # Batch size for validation
    shuffle=False  # Do not shuffle validation data
)


# ==============================
# Cell 7: Model, Optimizer, Scheduler
# ==============================

# Number of unique intent labels (classes)
num_labels = len(label_encoder.classes_)  # Determine number of classes from label encoder

# Load a pre-trained Transformer model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,  # Name of the pre-trained model
    num_labels=num_labels  # Number of output labels
)

# Move the model to the selected device (CPU or GPU)
model.to(device)  # Transfer model to device

# Initialize the AdamW optimizer
optimizer = AdamW(
    model.parameters(),  # Parameters of the model to optimize
    lr=LEARNING_RATE  # Learning rate
)

# Total number of training steps = number of batches * number of epochs
total_steps = len(train_loader) * EPOCHS  # Compute total steps

# Create a learning rate scheduler with linear warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer,  # Optimizer whose learning rate will be scheduled
    num_warmup_steps=0,  # Number of warmup steps (0 means no warmup)
    num_training_steps=total_steps  # Total number of training steps
)


# ==============================
# Cell 8: Training and Validation Loop
# ==============================

# Function to train the model for one epoch
def train_one_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()  # Set model to training mode
    total_loss = 0  # Initialize total loss for the epoch

    for batch in data_loader:  # Iterate over each batch of data
        input_ids = batch["input_ids"].to(device)  # Move token IDs to device
        attention_mask = batch["attention_mask"].to(device)  # Move attention mask to device
        labels = batch["labels"].to(device)  # Move labels to device

        optimizer.zero_grad()  # Clear previous gradients

        outputs = model(
            input_ids=input_ids,  # Token IDs as input to the model
            attention_mask=attention_mask,  # Attention masks
            labels=labels  # Ground truth labels for computing loss
        )

        loss = outputs.loss  # Extract loss from model outputs
        logits = outputs.logits  # Extract raw predictions (logits)

        total_loss += loss.item()  # Add the loss value to the total loss

        loss.backward()  # Backpropagate the loss to compute gradients

        optimizer.step()  # Update model parameters
        scheduler.step()  # Update learning rate according to scheduler

    avg_loss = total_loss / len(data_loader)  # Compute average loss over all batches
    return avg_loss  # Return the average loss


# Function to evaluate the model on validation (or test) data
def eval_model(model, data_loader, device):
    model.eval()  # Set model to evaluation mode
    preds = []  # List to store predicted labels
    true_labels = []  # List to store true labels
    total_loss = 0  # Total loss over evaluation set

    with torch.no_grad():  # Disable gradient computation for evaluation
        for batch in data_loader:  # Iterate over each batch
            input_ids = batch["input_ids"].to(device)  # Move token IDs to device
            attention_mask = batch["attention_mask"].to(device)  # Move attention mask to device
            labels = batch["labels"].to(device)  # Move labels to device

            outputs = model(
                input_ids=input_ids,  # Token IDs
                attention_mask=attention_mask,  # Attention masks
                labels=labels  # Ground truth labels for computing loss
            )

            loss = outputs.loss  # Extract loss
            logits = outputs.logits  # Extract logits

            total_loss += loss.item()  # Accumulate loss

            _, batch_preds = torch.max(logits, dim=1)  # Take argmax over logits to get predicted class indices

            preds.extend(batch_preds.cpu().numpy())  # Append predicted labels (move to CPU and convert to numpy)
            true_labels.extend(labels.cpu().numpy())  # Append true labels (move to CPU and convert to numpy)

    avg_loss = total_loss / len(data_loader)  # Compute average loss
    accuracy = accuracy_score(true_labels, preds)  # Compute accuracy
    # Use all possible labels from label_encoder and set zero_division=0 to handle missing classes in small batches
    report = classification_report(true_labels, preds, labels=label_encoder.transform(label_encoder.classes_), target_names=label_encoder.classes_, zero_division=0)  # Classification report

    return avg_loss, accuracy, report  # Return loss, accuracy, and report


# ==============================
# Cell 9: Run Training
# ==============================

best_val_accuracy = 0.0  # Track the best validation accuracy during training
BEST_MODEL_PATH = "models/best_model.pt"  # Path to save the best model

for epoch in range(EPOCHS):  # Loop over each epoch
    print(f"Epoch {epoch + 1}/{EPOCHS}")  # Print current epoch

    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)  # Train for one epoch
    print(f"Train loss: {train_loss:.4f}")  # Print training loss

    val_loss, val_accuracy, val_report = eval_model(model, val_loader, device)  # Evaluate on validation set
    print(f"Val loss: {val_loss:.4f}, Val accuracy: {val_accuracy:.4f}")  # Print validation loss and accuracy
    print("Validation classification report:")  # Print header for report
    print(val_report)  # Print full classification report

    # If this epoch's validation accuracy is the best so far, save the model
    if val_accuracy > best_val_accuracy:  # Check if current val accuracy is better
        best_val_accuracy = val_accuracy  # Update best validation accuracy
        torch.save(model.state_dict(), BEST_MODEL_PATH)  # Save the model's state dictionary
        print(f"New best model saved to {BEST_MODEL_PATH}")  # Notify user about saving best model

print(f"Best validation accuracy achieved: {best_val_accuracy:.4f}")  # Print best validation accuracy

Using device: cpu
Created/Updated example dataset at data/intent_questions.csv
                                             text                  label
0  How much urea should I use for wheat per acre?       fertilizer_wheat
1  What is the best way to control pests in rice?      pest_control_rice
2      How often should I irrigate my maize crop?       irrigation_maize
3       What fertilizer is recommended for paddy?        fertilizer_rice
4      How can I prevent fungal disease in wheat?  disease_control_wheat
Label mapping: {'disease_control_wheat': np.int64(0), 'fertilizer_rice': np.int64(1), 'fertilizer_wheat': np.int64(2), 'irrigation_maize': np.int64(3), 'irrigation_vegetables': np.int64(4), 'irrigation_wheat': np.int64(5), 'pest_control_cotton': np.int64(6), 'pest_control_maize': np.int64(7), 'pest_control_rice': np.int64(8)}
Train samples: 9, Val samples: 5, Test samples: 5
Saved label encoder to models/label_encoder.joblib


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 145.95it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3
Train loss: 2.1982
Val loss: 2.2280, Val accuracy: 0.0000
Validation classification report:
                       precision    recall  f1-score   support

disease_control_wheat       0.00      0.00      0.00       0.0
      fertilizer_rice       0.00      0.00      0.00       1.0
     fertilizer_wheat       0.00      0.00      0.00       1.0
     irrigation_maize       0.00      0.00      0.00       0.0
irrigation_vegetables       0.00      0.00      0.00       0.0
     irrigation_wheat       0.00      0.00      0.00       1.0
  pest_control_cotton       0.00      0.00      0.00       1.0
   pest_control_maize       0.00      0.00      0.00       1.0
    pest_control_rice       0.00      0.00      0.00       0.0

             accuracy                           0.00       5.0
            macro avg       0.00      0.00      0.00       5.0
         weighted avg       0.00      0.00      0.00       5.0

Epoch 2/3
Train loss: 2.1500
Val loss: 2.2314, Val accuracy: 0.0000
Validati